In [ ]:
import sys
import os
import copy
import shutil

import numpy as np

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.predictor import ShorelinePredictor

from src.data_processing.dataset_loader import CoastData

import json

In [ ]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
image_type_paths = {
    # "oblique": {
    #     "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/SCLabels_oblique_registered_v1.0.0/")),
    #     "num_classes": 3,
    #     "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/UNSW/oblique"))
    # },
    "rectified": {
        # "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/SCLabels_v1.0.0/")),
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/UNSW_Output_SCLabels_style/")),
        "num_classes": 3,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/UNSW/rectified"))
    }
}

networks: dict[str] = {
    "DeepLabV3": {
        # "oblique": {
        #     "fold_0": "2026-01-08-14-11-42_oblique_fold_0_DeepLabV3_512x512",
        # },
        "rectified": {
            "only_UNSW": "2026-10-27-16-32-17_DeepLabV3_512x512",
        }
    }
}

patches = {
    "512x512": {
        "patch_size": (512, 512),
        "stride": (256, 256)
    },
}

output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/UNSW/"))

In [ ]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for data_type in image_type_paths:
    print(f"\n{'#'*30}\nProcessing {data_type} images\n{'#'*30}")
    # Create folder
    output_subdir = os.path.join(output_dir, data_type)
    if not os.path.exists(output_subdir):
        os.makedirs(output_subdir)
        print(f"Created output directory: {output_subdir}")
        
    data_path = image_type_paths[data_type]["path"]
    num_classes = image_type_paths[data_type]["num_classes"]
    weights_path = image_type_paths[data_type]["weights_path"]
    
    print(f"Data path: {data_path}")

    # Load data
    data = CoastData(data_path)

    get_coords = False
    get_mask = True if data_type == "rectified" else False
    get_land_side = True
    get_original_image_path = True if data_type == "oblique" else False
    filtered_data = data.split_data(get_metadata=True, get_coords=get_coords, get_mask=get_mask, get_original_image_path=get_original_image_path)

    patch_name = "512x512"
    patch_info = patches[patch_name]

    print(f"Number of samples: {len(filtered_data['test']['images'])}")
    for network in networks:
        # for patch_name, patch_info in patches.items():
            # Create images/masks/predictions folders
        for fold in networks[network][data_type]:
            print(f"\n{'='*20}\nProcessing fold: {fold}\n{'='*20}")
            base_output_dir = os.path.join(output_subdir, f"{fold}_{network}_{patch_name}")
            print(f"Creating output directory: {base_output_dir}")
            images_output_dir = os.path.join(base_output_dir, "images")
            masks_output_dir = os.path.join(base_output_dir, "masks")
            predictions_output_dir = os.path.join(base_output_dir, "predictions")
            predicted_mask_output_dir = os.path.join(base_output_dir, "predicted_masks")

            for dir_path in [images_output_dir, masks_output_dir, predictions_output_dir, predicted_mask_output_dir]:
                if not os.path.exists(dir_path):
                    os.makedirs(dir_path)
                    print(f"Created directory: {dir_path}")

            print(f"\n{'-'*20}\nPredicting with {network} - {data_type}\n{'-'*20}")
            net_weights_path = os.path.join(weights_path, networks[network][data_type][fold], "models/best_model.pth")

            all_metadata_list = []
            
            predictor = ShorelinePredictor(network, net_weights_path, num_classes)

            counter = 0
            total_images = len(filtered_data['test']['images'])

            # Predict only the test set
            for path_img, path_mask, metadata in zip(filtered_data['test']['images'], filtered_data['test']['masks'], filtered_data['test']['metadata']):
                # Get filenames
                img_filename = os.path.basename(path_img)
                mask_filename = img_filename.replace("image", "mask")
                output_img_path = os.path.join(images_output_dir, img_filename)
                output_mask_path = os.path.join(masks_output_dir, mask_filename)
                # print("Path mask:", path_mask)
                
                # Predict
                shoreline_coords = metadata['image']['shoreline']['coordinates']
                landward_pixel_pred = 1 if data_type == "rectified" else 0
                seaward_pixel_pred = 2 if data_type == "rectified" else 1
                output = predictor.predict_oblique_with_coords(path_img, shoreline_coords=shoreline_coords, patch_size=patch_info["patch_size"], stride=patch_info["stride"], extract_mask_coords=True, landward_pixel_pred=landward_pixel_pred, seaward_pixel_pred=seaward_pixel_pred)
                # Copy original image and mask to output folders
                if not os.path.exists(output_img_path):
                    shutil.copy(path_img, output_img_path)
                if not os.path.exists(output_mask_path):
                    shutil.copy(path_mask, output_mask_path)
                # Save prediction and predicted mask
                prediction_filename = mask_filename.replace("mask", "prediction")
                output_prediction_path =  os.path.join(predictions_output_dir, prediction_filename)
                plt.imsave(output_prediction_path, output['predicted_image'])

                predicted_mask_filename = mask_filename.replace("mask", "predicted_mask")
                output_predicted_mask_path =  os.path.join(predicted_mask_output_dir, predicted_mask_filename)
                plt.imsave(output_predicted_mask_path, output['predicted_mask'], cmap='gray')

                # Update metadata
                meta_copy = copy.deepcopy(metadata)
                meta_copy.setdefault("image", {})
                meta_copy["image"].setdefault("shoreline", {})

                patch = networks.get(network, {}).get('patch', {})
                patch_size = patch_info.get("patch_size", "Unknown")
                stride = patch_info.get("stride", "Unknown")

                meta_copy['image']['predicted_mask'] = {
                    "filename": predicted_mask_filename,
                    "description": f"Patch size {patch_size} and stride {stride}",
                    "labels": {
                        "0": "NoData",
                        "75": "Landwards",
                        "150": "Seawards",
                        "255": "Shoreline"
                    }
                }

                meta_copy['image']['prediction'] = {
                    "filename": prediction_filename,
                    "description": f"Predicted shoreline overlaid on original image by model (e.g., {network}), patched prediction with patch size {patch_size} and stride {stride}. Original shoreline shown in green, predicted shoreline shown in red and overlapping areas in yellow."
                }

                meta_copy["image"]["shoreline"]["coordinates"] = {
                    "u": output.get('original_shoreline_coords', {}).get('u', []),
                    "v": output.get('original_shoreline_coords', {}).get('v', [])
                }
                meta_copy["image"]["shoreline"]["description"] = "uv coordinates of the shoreline..."

                meta_copy['image']['predicted_shoreline'] = {
                    "description": f"PREDICTED shoreline by model (e.g., {network}), patched prediction with patch size {patch_size} and stride {stride}.",
                    "coordinates": {
                        "u": output.get('shoreline_coords', {}).get('u', []),
                        "v": output.get('shoreline_coords', {}).get('v', [])
                    }
                }

                all_metadata_list.append(meta_copy)

                counter += 1
                if counter % max(1, total_images // 10) == 0 or counter == total_images:
                    print(f"Processed {counter}/{total_images} images ({(counter/total_images)*100:.1f}%)")

            # Save metadata to JSON
            metadata_output_path = os.path.join(base_output_dir, "metadata.json")
            with open(metadata_output_path, 'w') as json_file:
                json.dump(all_metadata_list, json_file, indent=4)